In [ ]:
import pandas as pd

# Load the CSV file into a DataFrame
df = pd.read_csv('hand_landmarks.csv')

# Show the first few rows of the DataFrame
df.head()


In [ ]:
# Group by timestamp to process each frame separately
grouped_frames = df.groupby('timestamp')

# Inspect the first frame (group of landmarks)
first_frame = grouped_frames.get_group(df['timestamp'].unique()[0])
first_frame.head()


In [ ]:
import numpy as np

# Function to calculate the angle between three points (landmarks)
def calculate_angle(a, b, c):
    v1 = np.array([a[0] - b[0], a[1] - b[1], a[2] - b[2]])  # Vector AB
    v2 = np.array([c[0] - b[0], c[1] - b[1], c[2] - b[2]])  # Vector BC
    
    # Dot product and magnitudes
    dot_product = np.dot(v1, v2)
    magnitude_v1 = np.linalg.norm(v1)
    magnitude_v2 = np.linalg.norm(v2)
    
    # Avoid division by zero
    if magnitude_v1 == 0 or magnitude_v2 == 0:
        return None
    
    # Angle in radians
    angle_radians = np.arccos(dot_product / (magnitude_v1 * magnitude_v2))
    
    # Convert radians to degrees
    angle_degrees = np.degrees(angle_radians)
    
    return angle_degrees


In [ ]:
# Function to calculate the speed of landmarks (xyz) the hand
def calculate_speed(df):
    # Calculate the speed of each landmark (xyz) using the difference between consecutive frames
    speed = df[['x', 'y', 'z']].diff().abs()
    
    # Calculate the total speed of the hand
    speed['total'] = speed.sum(axis=1)
    
    return speed


In [ ]:
# Put the landmarks into marker_positions variable
marker_positions = first_frame[['x', 'y', 'z']].values

In [ ]:
# Calculate the speed of the hand
speed = calculate_speed(first_frame)
speed.head()

In [ ]:
# Function to calculate angles for a frame
def calculate_finger_angles(frame):
    angles = {}
    
    # Thumb (landmarks 1-4)
    thumb_angle = calculate_angle(frame.loc[1, ['x', 'y', 'z']],
                                  frame.loc[2, ['x', 'y', 'z']],
                                  frame.loc[3, ['x', 'y', 'z']])
    angles['Thumb'] = thumb_angle
    
    # Index finger (landmarks 5-8)
    index_angle = calculate_angle(frame.loc[5, ['x', 'y', 'z']],
                                  frame.loc[6, ['x', 'y', 'z']],
                                  frame.loc[7, ['x', 'y', 'z']])
    angles['Index'] = index_angle
    
    # Middle finger (landmarks 9-12)
    middle_angle = calculate_angle(frame.loc[9, ['x', 'y', 'z']],
                                   frame.loc[10, ['x', 'y', 'z']],
                                   frame.loc[11, ['x', 'y', 'z']])
    angles['Middle'] = middle_angle
    
    # Ring finger (landmarks 13-16)
    ring_angle = calculate_angle(frame.loc[13, ['x', 'y', 'z']],
                                 frame.loc[14, ['x', 'y', 'z']],
                                 frame.loc[15, ['x', 'y', 'z']])
    angles['Ring'] = ring_angle
    
    # Pinky finger (landmarks 17-20)
    pinky_angle = calculate_angle(frame.loc[17, ['x', 'y', 'z']],
                                  frame.loc[18, ['x', 'y', 'z']],
                                  frame.loc[19, ['x', 'y', 'z']])
    angles['Pinky'] = pinky_angle
    
    return angles

# Loop through frames and calculate angles for each frame
all_angles = {}
for timestamp, frame in grouped_frames:
    all_angles[timestamp] = calculate_finger_angles(frame)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Assuming all_angles and marker_positions are already defined
# all_angles: Dictionary of angles with timestamps as keys
# marker_positions: Dictionary of marker positions with timestamps as keys

# Convert angles data into a DataFrame
angles_df = pd.DataFrame(all_angles).T  # Transpose to make timestamps the index

# Convert marker positions data into a DataFrame
marker_positions_df = pd.DataFrame(marker_positions).T  # Transpose to make timestamps the index

# Convert speed data into a DataFrame
speed_df = pd.DataFrame(speed['total']).T  # Transpose to make timestamps the index

# Plot the angles over time
plt.figure(figsize=(10, 6))
for finger in angles_df.columns:
    plt.plot(angles_df.index, angles_df[finger], label=finger)

plt.title('Finger Joint Angles Over Time')
plt.xlabel('Timestamp')
plt.ylabel('Angle (degrees)')
plt.legend()
plt.show()

# Plot the trajectory of the marker
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot(marker_positions_df['x'], marker_positions_df['y'], marker_positions_df['z'], label='Marker Trajectory')

ax.set_title('Marker Trajectory Over Time')
ax.set_xlabel('X Position')
ax.set_ylabel('Y Position')
ax.set_zlabel('Z Position')
ax.legend()
plt.show()

# Plot the speed of the hand over time
plt.figure(figsize=(10, 6))
plt.plot(speed_df.index, speed_df['total'])
plt.title('Hand Speed Over Time')
plt.xlabel('Timestamp')
plt.ylabel('Speed')
plt.show()

# Save the processed data to CSV files
angles_df.to_csv('finger_angles.csv')
marker_positions_df.to_csv('marker_positions.csv')
speed_df.to_csv('hand_speed.csv')